**Responsável : Jorge Tavares**

**LangGraph Agent**

Este projeto explora a construção de um agente de IA utilizando LangGraph, LangChain e Gemini.

A ideia é criar um agente capaz de conversar com o usuário e, quando necessário, utilizar ferramentas para ler e escrever arquivos de texto. O LangGraph é utilizado para organizar esse fluxo, permitindo que o agente decida quando deve responder diretamente ou executar uma ferramenta.
O projeto foi desenvolvido como uma forma prática de entender os principais conceitos por trás da construção de AI Agents com LLMs e ferramentas.

In [3]:
#Instala ou atualiza as bibliotecas usadas no projeto
!pip install -U langgraph langchain langchain-google-genai

In [2]:
from google.colab import userdata

# A chave que está armazenada nos Secrets do Colab
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

In [7]:
# Importando o Gemini para usar como modelo de linguagem
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    google_api_key=GEMINI_API_KEY
)


# Imports que vamos usar para montar o agente com LangGraph
from typing import Annotated, TypedDict

from langchain.tools import tool
from langchain_core.messages import HumanMessage

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition


# Criando o estado do agente
# Aqui vamos guardar o histórico da conversa
class State(TypedDict):
    messages: Annotated[list, add_messages]


# Criando uma ferramenta para escrever arquivos
@tool
def escrever_arquivo(nome_arquivo: str, conteudo: str) -> str:
    """Cria um arquivo de texto com o conteúdo recebido."""

    # Abre o arquivo e escreve o conteúdo nele
    with open(nome_arquivo, "w", encoding="utf-8") as arquivo:
        arquivo.write(conteudo)

    return f"Arquivo '{nome_arquivo}' criado com sucesso."


# Criando uma ferramenta para ler arquivos
@tool
def ler_arquivo(nome_arquivo: str) -> str:
    """Lê o conteúdo de um arquivo de texto."""

    try:
        # Tenta abrir o arquivo para fazer a leitura
        with open(nome_arquivo, "r", encoding="utf-8") as arquivo:
            conteudo = arquivo.read()

        return conteudo

    except FileNotFoundError:

        # Caso o arquivo não exista, retornamos uma mensagem informando o erro
        return f"O arquivo '{nome_arquivo}' não foi encontrado."


# Reunindo as ferramentas que o agente pode utilizar
tools = [
    escrever_arquivo,
    ler_arquivo
]


# Informando ao Gemini quais ferramentas ele pode utilizar
llm_with_tools = llm.bind_tools(tools)


# Função principal do chatbot
def chatbot(state: State):

    # Recuperamos o histórico da conversa
    messages = state["messages"]

    # Enviamos o histórico para o Gemini e esperamos uma resposta
    response = llm_with_tools.invoke(messages)

    # Adicionamos a resposta ao estado
    return {
        "messages": [response]
    }


# Criando o nó responsável por executar as ferramentas
tool_node = ToolNode(tools=tools)


# Criando o grafo que vai controlar o fluxo do agente
builder = StateGraph(State)


# Adicionando o chatbot e as ferramentas ao grafo
builder.add_node("chatbot", chatbot)
builder.add_node("tools", tool_node)


# O fluxo começa pelo chatbot
builder.add_edge(START, "chatbot")


# Depois da resposta do chatbot, o LangGraph verifica
# se é necessário utilizar alguma ferramenta
builder.add_conditional_edges(
    "chatbot",
    tools_condition
)


# Depois de executar uma ferramenta,
# o resultado volta para o chatbot
builder.add_edge("tools", "chatbot")


# Compilando o grafo para transformar essa estrutura em um agente executável
graph = builder.compile()


# Criando o estado inicial da conversa
state = {
    "messages": []
}


# Mantém o agente funcionando até o usuário decidir sair
while True:

    pergunta = input("\nVocê: ")

    # Permite encerrar a conversa
    if pergunta.lower() in ["sair", "exit", "quit"]:
        print("\nAgente encerrado.")
        break

    # Adiciona a pergunta ao histórico
    state["messages"].append(
        HumanMessage(content=pergunta)
    )

    # Executa o fluxo do agente
    state = graph.invoke(state)

    # Mostra somente o texto da resposta
    resposta = state["messages"][-1].content

    # Pega somente o texto da resposta
    if isinstance(resposta, list):
        resposta = "".join(
            item["text"]
            for item in resposta
            if item.get("type") == "text"
        )

    print(f"\nAgente: {resposta}")


Você: O que você consegue fazer ?

Agente: Olá! Eu sou um assistente de inteligência artificial e posso ajudar você em diversas tarefas. Aqui estão algumas das principais coisas que posso fazer:

---

### 1. **Manipulação de Arquivos de Texto**
Neste ambiente, tenho ferramentas específicas para lidar com arquivos:
* **Criar/Escrever em arquivos (`escrever_arquivo`):** Posso gerar e salvar documentos de texto, arquivos de código, listas, relatórios, etc.
* **Ler arquivos (`ler_arquivo`):** Posso ler o conteúdo de arquivos de texto existentes para analisar, resumir, revisar ou extrair informações.

---

### 2. **Processamento de Texto e Redação**
* **Redigir documentos:** Artigos, e-mails, relatórios, resumos, cartas formais ou textos criativos (histórias, poemas).
* **Revisão e Correção:** Corrigir gramática, ortografia, estilo e clareza de textos em diversos idiomas.
* **Tradução:** Traduzir conteúdos entre múltiplos idiomas.

---

### 3. **Programação e Tecnologia**
* **Escrever códi